=============================================================================
PREMIER LEAGUE AI PREDICTOR
Phase 1 — Data Pipeline & Feature Store
=============================================================================
Steps:
  1. Pull 10 seasons of PL results via football-data.org
  2. Scrape xG / xGA / PPDA from Understat
  3. Compute rolling features (5 & 10 match windows)
  4. ELO ratings (standard + goal-adjusted)
  5. Pi-ratings (home/away attack & defence splits)
  6. Historical odds from football-data.co.uk
  7. Live odds from The Odds API
  8. Merge everything → SQLite + Parquet feature store

Setup:
  pip install requests pandas pyarrow aiohttp understat tqdm python-dotenv

API Keys (free):
  football-data.org  → https://www.football-data.org/client/register
  The Odds API       → https://the-odds-api.com/#get-access

  Either set them in a .env file:
      FOOTBALL_DATA_KEY=your_key_here
      ODDS_API_KEY=your_key_here

  Or paste directly into the CONFIG section below.
=============================================================================

In [ ]:
import os
import time
import sqlite3
import asyncio
import warnings
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Try loading .env if python-dotenv is available
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


# =============================================================================
# CONFIG — edit this section
# =============================================================================

FOOTBALL_DATA_KEY = os.getenv("FOOTBALL_DATA_KEY", "YOUR_KEY_HERE")
ODDS_API_KEY      = os.getenv("ODDS_API_KEY",      "YOUR_KEY_HERE")

# 10 seasons of Premier League data
SEASONS = [
    "2014-15", "2015-16", "2016-17", "2017-18", "2018-19",
    "2019-20", "2020-21", "2021-22", "2022-23", "2023-24",
]

BASE_DIR  = Path(".")
DATA_DIR  = BASE_DIR / "data"
RAW_DIR   = DATA_DIR / "raw"
FEAT_DIR  = DATA_DIR / "features"
DB_PATH   = DATA_DIR / "pl_predictor.db"

In [ ]:
# =============================================================================
# SECTION 0 — Directory setup
# =============================================================================

def setup_dirs():
    for d in [DATA_DIR, RAW_DIR, FEAT_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    print(f"Directories ready. DB will be at: {DB_PATH.resolve()}")


# =============================================================================
# SECTION 1 — Pull results from football-data.org
# =============================================================================

FOOTBALL_DATA_BASE = "https://api.football-data.org/v4"
COMPETITION        = "PL"

# Maps football-data.org long names → canonical short names
# Understat and football-data.co.uk both use the short versions
TEAM_NAME_MAP = {
    "Arsenal FC":                    "Arsenal",
    "Aston Villa FC":                "Aston Villa",
    "Brentford FC":                  "Brentford",
    "Brighton & Hove Albion FC":     "Brighton",
    "Burnley FC":                    "Burnley",
    "Chelsea FC":                    "Chelsea",
    "Crystal Palace FC":             "Crystal Palace",
    "Everton FC":                    "Everton",
    "Fulham FC":                     "Fulham",
    "Ipswich Town FC":               "Ipswich",
    "Leeds United FC":               "Leeds",
    "Leicester City FC":             "Leicester",
    "Liverpool FC":                  "Liverpool",
    "Luton Town FC":                 "Luton",
    "Manchester City FC":            "Manchester City",
    "Manchester United FC":          "Manchester United",
    "Middlesbrough FC":              "Middlesbrough",
    "Newcastle United FC":           "Newcastle",
    "Norwich City FC":               "Norwich",
    "Nottingham Forest FC":          "Nottingham Forest",
    "Queens Park Rangers FC":        "QPR",
    "Sheffield United FC":           "Sheffield United",
    "Southampton FC":                "Southampton",
    "Stoke City FC":                 "Stoke",
    "Sunderland AFC":                "Sunderland",
    "Swansea City AFC":              "Swansea",
    "Tottenham Hotspur FC":          "Tottenham",
    "Watford FC":                    "Watford",
    "West Bromwich Albion FC":       "West Brom",
    "West Ham United FC":            "West Ham",
    "Wolverhampton Wanderers FC":    "Wolves",
    "Hull City AFC":                 "Hull",
    "AFC Bournemouth":               "Bournemouth",
}


def fetch_season_matches(season_str: str) -> pd.DataFrame:
    """
    Fetch all finished PL matches for one season from football-data.org.
    season_str: e.g. '2023-24'
    """
    start_year = int(season_str.split("-")[0])
    url    = f"{FOOTBALL_DATA_BASE}/competitions/{COMPETITION}/matches"
    params = {"season": start_year}
    headers = {"X-Auth-Token": FOOTBALL_DATA_KEY}

    resp = requests.get(url, headers=headers, params=params, timeout=15)

    if resp.status_code == 429:
        print("  Rate limited — waiting 65s...")
        time.sleep(65)
        resp = requests.get(url, headers=headers, params=params, timeout=15)

    resp.raise_for_status()
    matches = resp.json().get("matches", [])

    rows = []
    for m in matches:
        ft = m.get("score", {}).get("fullTime", {})
        ht = m.get("score", {}).get("halfTime", {})
        rows.append({
            "match_id":       m["id"],
            "season":         season_str,
            "matchday":       m.get("matchday"),
            "date":           m.get("utcDate", "")[:10],
            "status":         m.get("status"),
            "home_team":      m["homeTeam"]["name"],
            "away_team":      m["awayTeam"]["name"],
            "home_goals_ft":  ft.get("home"),
            "away_goals_ft":  ft.get("away"),
            "home_goals_ht":  ht.get("home"),
            "away_goals_ht":  ht.get("away"),
            "winner":         m.get("score", {}).get("winner"),
            "referee":        m.get("referees", [{}])[0].get("name") if m.get("referees") else None,
            "venue":          m.get("venue"),
        })

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    return df[df["status"] == "FINISHED"].copy()


def pull_all_results() -> pd.DataFrame:
    print("\n── STEP 1: Pulling results from football-data.org ──────────────")
    print("Free tier = 10 calls/min → sleeping 7s between seasons")

    all_dfs = []
    for season in tqdm(SEASONS, desc="Seasons"):
        print(f"  {season}...", end=" ", flush=True)
        try:
            df_s = fetch_season_matches(season)
            all_dfs.append(df_s)
            print(f"{len(df_s)} matches")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(7)

    df = pd.concat(all_dfs, ignore_index=True)
    df["home_team"] = df["home_team"].replace(TEAM_NAME_MAP)
    df["away_team"] = df["away_team"].replace(TEAM_NAME_MAP)

    # Season year for joining with Understat
    df["season_year"] = df["date"].dt.year.where(
        df["date"].dt.month >= 8, df["date"].dt.year - 1
    )

    # Points and goal diff
    df["home_points"] = df.apply(
        lambda r: 3 if r.home_goals_ft > r.away_goals_ft else
                  (1 if r.home_goals_ft == r.away_goals_ft else 0), axis=1
    )
    df["away_points"] = df.apply(
        lambda r: 3 if r.away_goals_ft > r.home_goals_ft else
                  (1 if r.away_goals_ft == r.home_goals_ft else 0), axis=1
    )
    df["goal_diff_home"] = df["home_goals_ft"] - df["away_goals_ft"]
    df["result"] = df.apply(
        lambda r: "H" if r.home_goals_ft > r.away_goals_ft else
                  ("A" if r.home_goals_ft < r.away_goals_ft else "D"), axis=1
    )

    df = df.sort_values("date").reset_index(drop=True)
    df.to_parquet(RAW_DIR / "results_raw.parquet", index=False)

    print(f"\n  ✅ {len(df)} finished matches across {df.season.nunique()} seasons")
    return df

